Import gw data analyzed

In [13]:
import os
import pandas as pd

# Load the csv file
league_name = 'balo' #'rpk', 'ifc', 'rbsc'

In [14]:
def get_dim_manager(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"
    dim_manager_path = os.path.join(folder_path, "dim_managers.csv")
    return pd.read_csv(dim_manager_path)

def get_df(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"

    # 2. Load all available files (assuming up to gw36 are available right now)
    # We loop from 1 to 36 to build the full historic dataset
    file_names = [
        os.path.join(folder_path, f"gw{str(i).zfill(2)}_analyzed.csv")
        for i in range(1, 38+1)
    ]

    # Filter out files that don't exist yet (safeguard for future weeks like gw37, gw38)
    existing_files = [f for f in file_names if os.path.exists(f)]
    if not existing_files:
        raise FileNotFoundError(
            f"No analyzed CSV files found in directory: {folder_path}"
        )

    # 3. Read and combine all found CSV files into one master DataFrame
    df = pd.concat([pd.read_csv(file) for file in existing_files], ignore_index=True)

    # 4. Join and get manager info
    dim_manager_df = get_dim_manager(league_name)
    df = df.merge(
        dim_manager_df[["id", "player_first_name", "player_last_name", "name"]],
        left_on="manager_id", right_on="id",
        how="left"
    )

    column_order = [
        'manager_id',
        'id',
        'player_first_name',
        'player_last_name',
        'name',
        'gw_no',
        'points',
        'transfers_cost',
        'active_chip',
        'points_on_bench',
        'h2h_points',
        'rank',
        'pnl',
    ]

    return df[column_order]

In [15]:
df = get_df(league_name)

In [16]:
df

,manager_id,id,player_first_name,player_last_name,name,gw_no,points,transfers_cost,active_chip,points_on_bench,h2h_points,rank,pnl
0,4062893,4062893,Poomchai,Chotichaicharin,FloWirtz,1,65,0,NaN,13,65.0,1,500
1,7496060,7496060,Luckyman,POLO,Lucky Cookie,1,64,0,NaN,7,64.0,2,0
2,3209933,3209933,Oak,P,Oak,1,62,0,NaN,7,62.0,3,0
3,3357715,3357715,Pear,The Richest,Pear The Richest,1,61,0,NaN,19,61.0,4,0
4,3459094,3459094,BooM,Suriyabhivadh,69 united,1,59,0,NaN,9,59.0,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,3646810,3646810,Napin,Surakkhaka,snapin,38,46,0,NaN,0,46.0,11,0
604,4878022,4878022,Tat,Koochingchai,Highbury,38,43,0,NaN,0,43.0,12,0
605,7372130,7372130,bankky,bank,Gunner the champion,38,47,4,bboost,0,43.0,12,0
606,8005763,8005763,num,dam,Num559 fc,38,41,0,NaN,0,41.0,13,0


In [17]:
# check 38 entries for every player
for no_of_entries in df.groupby('manager_id')['h2h_points'].count():
    assert no_of_entries == 38

In [18]:
def get_player_history(df, manager_id, gw_start=1, gw_end=38, chronological=True):
    """Extracts and filters the gameweek history for a single manager."""
    # 1. Filter for the specific manager and gameweek range
    player_df = df[
        (df["manager_id"] == manager_id)
        & (df["gw_no"] >= gw_start)
        & (df["gw_no"] <= gw_end)
    ].copy()

    # 2. Sort by gameweek order
    sort_order = True if chronological else False
    player_df = player_df.sort_values(by="gw_no", ascending=sort_order)

    # 3. Reorder columns to make it readable as a personal timeline
    column_order = [
        "gw_no",
        "rank",
        "h2h_points",
        "points",
        "points_on_bench",
        "transfers_cost",
        "active_chip",
        "pnl",
    ]

    # Clean up the index so it looks like a fresh dataframe (0, 1, 2...)
    return player_df[column_order].reset_index(drop=True)

In [19]:
# get_player_history(df, 1357701)

In [20]:
def summarize_league(league_name, gw_start=1, gw_end=38):
    """Loads all available CSV files for a specific league, computes performance

    statistics, and filters the final summary by the requested gameweek range.
    """
    df = get_df(league_name)

    # 4. Calculate Weekly Winners and Losers across the whole dataset
    max_ranks = df.groupby("gw_no")["rank"].transform("max")
    df["is_weekly_winner"] = df["rank"] == 1
    if league_name == "rbsc":
        df["is_weekly_loser"] = df["rank"] == max_ranks
    else:
        df["is_weekly_loser"] = df["rank"] >= max_ranks - 2

    # 5. NOW apply your custom gameweek range filter
    df_filtered = df[(df["gw_no"] >= gw_start) & (df["gw_no"] <= gw_end)].copy()

    # 6. Aggregate metrics per manager for the filtered range
    summary = (
        df_filtered.groupby("manager_id")
        .agg(
            total_h2h_points=("h2h_points", "sum"),
            total_points=("points", "sum"),
            bench_points=("points_on_bench", "sum"),
            total_pnl=("pnl", "sum"),
            weekly_wins=("is_weekly_winner", "sum"),
            weekly_losses=("is_weekly_loser", "sum"),
        )
        .reset_index()
    )

    # 7. Add Dense Rank based on H2H points within this specific window
    summary["overall_rank"] = (
        summary["total_h2h_points"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    # 8. Add Meta info to keep track of the slice
    summary["gw_start"] = gw_start
    summary["gw_end"] = gw_end

    # 9. Dynamically load and merge Manager Names from the same folder
    dim_manager_df = get_dim_manager(league_name)
    summary = summary.merge(
        dim_manager_df[["id", "player_first_name", "player_last_name", "name"]],
        left_on="manager_id", right_on="id",
        how="left"
    )

    # 10. Clean layout order
    column_order = [
        "gw_start",
        "gw_end",
        "overall_rank",
        "manager_id",
        "name",
        "player_first_name",
        "player_last_name",
        "total_h2h_points",
        "total_points",
        "bench_points",
        "total_pnl",
        "weekly_wins",
        "weekly_losses",
    ]

    return summary[column_order].sort_values("overall_rank")

In [21]:
summary = summarize_league(league_name)

In [22]:
summary

,gw_start,gw_end,overall_rank,manager_id,name,player_first_name,player_last_name,total_h2h_points,total_points,bench_points,total_pnl,weekly_wins,weekly_losses
3,1,38,1,2369208,Boberry,BO,BERRY,2297.0,2305,401,1250,2,3
6,1,38,2,3459094,69 united,BooM,Suriyabhivadh,2271.0,2295,325,500,3,7
1,1,38,3,870769,9IX,S,S,2267.0,2271,312,500,2,5
12,1,38,4,4878022,Highbury,Tat,Koochingchai,2213.0,2241,343,1000,2,5
8,1,38,5,3961989,kiwi united,kavee,talomsin,2209.0,2237,300,1000,5,10
9,1,38,6,4062893,FloWirtz,Poomchai,Chotichaicharin,2196.0,2204,261,2000,5,8
4,1,38,7,3209933,Oak,Oak,P,2192.0,2192,300,500,2,10
2,1,38,8,1029833,Bramley Moyes,Oukas,Thomson,2188.0,2192,353,1750,6,3
0,1,38,9,556371,Fame's Squad,Pimadej,Siwapornpitak,2167.0,2175,383,-500,2,9
13,1,38,10,7372130,Gunner the champion,bankky,bank,2136.0,2180,364,-1000,1,8


In [23]:
df.to_csv(f"fact_gw_{league_name}.csv", index=False)

In [24]:
summary.to_csv(f"summary_{league_name}.csv", index=False)

Add end of season prize to summary (RPK)

In [19]:
# add end of season prize 
def add_eos_prize(summary, prize: dict):
    summary_copy = summary.copy()
    # assign prize to rank
    summary_copy['end_of_season_prize'] = summary['overall_rank'].map(prize).fillna(0).astype(int)
    #    assert summary_copy['end_of_season_prize'] == 0 # not applicable to ifc league because everybody paid upfront
    summary_copy['pnl'] = summary_copy['total_pnl'] + summary_copy['end_of_season_prize']
    return summary_copy

In [20]:
# add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # ifc
add_eos_prize(summary, {1:3000, 2:2000, 3:1000, 6:-1000, 7:-2000, 8:-3000}) # rpk
# add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # rbsc

,gw_start,gw_end,overall_rank,manager_id,name,player_first_name,player_last_name,total_h2h_points,total_points,bench_points,total_pnl,weekly_wins,weekly_losses,end_of_season_prize,pnl
0,1,38,1,294329,Aekk72,Atthapon,Parkart,2301,2305,323,0,6,15,3000,3000
1,1,38,2,967075,Victory Goalkeres,sirawat,dulyavit,2251,2259,250,-200,5,17,2000,1800
4,1,38,3,2412148,ngnteam,Tawiwut,Charuwat,2214,2214,395,2650,13,15,1000,3650
3,1,38,4,1369948,Morty FC,Pattapong,Charoenchaipong,2065,2073,344,-1300,4,26,0,-1300
2,1,38,5,1357701,OnkaewmaneeN,Nithiz,Onkaewmanee,1875,1999,463,-50,7,24,0,-50
5,1,38,6,6149266,pairyn FC,Natthawat,Charoenkitmongkol,1870,1870,74,-1100,5,29,-1000,-2100
